In [ ]:
# ============================================================
# Train-Validation(-Test) Split Integrity Check
# 1) Index-level overlap check (no sample appears in 2+ splits)
# 2) Duplicate / near-duplicate image check across splits
#    (perceptual hash -- catches exact + visually-identical copies,
#    which is common in medical image datasets with repeated slides)
# 3) Class distribution consistency (stratification check)
# 4) Total count sanity check
# ============================================================

!pip install -q imagehash
import imagehash
from PIL import Image
from collections import Counter

# ---- 0) recover the underlying index sets for train/val/test ----
def get_indices_and_paths(loader):
    ds = loader.dataset
    if hasattr(ds, "indices"):  # torch.utils.data.Subset
        base = ds.dataset
        idxs = list(ds.indices)
    else:
        base = ds
        idxs = list(range(len(ds)))
    paths = [base.samples[i][0] for i in idxs]   # ImageFolder stores (path, label) in .samples
    labels = [base.samples[i][1] for i in idxs]
    return set(idxs), paths, labels

train_idx_set, train_paths, train_labels = get_indices_and_paths(train_loader)
val_idx_set, val_paths, val_labels = get_indices_and_paths(val_loader)
test_idx_set, test_paths, test_labels = get_indices_and_paths(test_loader)

print(f"Train: {len(train_idx_set)} samples | Val: {len(val_idx_set)} samples | Test: {len(test_idx_set)} samples")
print(f"Total: {len(train_idx_set) + len(val_idx_set) + len(test_idx_set)}")


# ============================================================
# 1) INDEX-LEVEL OVERLAP CHECK
# ============================================================
overlap_train_val = train_idx_set & val_idx_set
overlap_train_test = train_idx_set & test_idx_set
overlap_val_test = val_idx_set & test_idx_set

print("\n=== 1) Index-Level Overlap Check ===")
print(f"Train ∩ Val overlap:  {len(overlap_train_val)} samples  {'⚠️ LEAKAGE!' if overlap_train_val else '✅ Clean'}")
print(f"Train ∩ Test overlap: {len(overlap_train_test)} samples  {'⚠️ LEAKAGE!' if overlap_train_test else '✅ Clean'}")
print(f"Val ∩ Test overlap:   {len(overlap_val_test)} samples  {'⚠️ LEAKAGE!' if overlap_val_test else '✅ Clean'}")


# ============================================================
# 2) DUPLICATE / NEAR-DUPLICATE IMAGE CHECK (perceptual hash)
#    Catches: exact file duplicates, re-saved/re-compressed
#    copies, and visually-identical slides accidentally split
#    across train/val/test.
# ============================================================
def compute_hashes(paths, hash_size=8):
    hashes = {}
    for p in paths:
        try:
            with Image.open(p) as img:
                h = imagehash.phash(img.convert("RGB"), hash_size=hash_size)
            hashes[p] = h
        except Exception as e:
            print(f"  Could not hash {p}: {e}")
    return hashes

print("\n=== 2) Perceptual Hash Duplicate Check (this may take a few minutes) ===")
train_hashes = compute_hashes(train_paths)
val_hashes = compute_hashes(val_paths)
test_hashes = compute_hashes(test_paths)

def find_cross_split_duplicates(hashes_a, hashes_b, name_a, name_b, threshold=2):
    """threshold: max Hamming distance to count as a near-duplicate (0 = exact match)."""
    duplicates = []
    hash_list_b = list(hashes_b.items())
    for path_a, h_a in hashes_a.items():
        for path_b, h_b in hash_list_b:
            if (h_a - h_b) <= threshold:   # imagehash supports subtraction = Hamming distance
                duplicates.append({
                    "Split A": name_a, "Path A": path_a,
                    "Split B": name_b, "Path B": path_b,
                    "Hamming Distance": h_a - h_b,
                })
    return duplicates

dup_train_val = find_cross_split_duplicates(train_hashes, val_hashes, "Train", "Val")
dup_train_test = find_cross_split_duplicates(train_hashes, test_hashes, "Train", "Test")
dup_val_test = find_cross_split_duplicates(val_hashes, test_hashes, "Val", "Test")

all_duplicates = dup_train_val + dup_train_test + dup_val_test
dup_df = pd.DataFrame(all_duplicates)

print(f"Train-Val near-duplicates (Hamming<=2):  {len(dup_train_val)}  {'⚠️' if dup_train_val else '✅'}")
print(f"Train-Test near-duplicates (Hamming<=2): {len(dup_train_test)}  {'⚠️' if dup_train_test else '✅'}")
print(f"Val-Test near-duplicates (Hamming<=2):   {len(dup_val_test)}  {'⚠️' if dup_val_test else '✅'}")

if len(dup_df) > 0:
    dup_df.to_csv("/kaggle/working/cross_split_duplicates.csv", index=False)
    print("\nSample duplicate pairs found:")
    print(dup_df.head(10).to_string(index=False))
else:
    print("\n✅ No cross-split duplicate/near-duplicate images detected.")


# ============================================================
# 3) CLASS DISTRIBUTION CONSISTENCY (stratification check)
# ============================================================
def class_dist_pct(labels):
    c = Counter(labels)
    total = sum(c.values())
    return {CLASS_NAMES[k]: round(v / total * 100, 2) for k, v in sorted(c.items())}

dist_df = pd.DataFrame({
    "Train (%)": class_dist_pct(train_labels),
    "Val (%)": class_dist_pct(val_labels),
    "Test (%)": class_dist_pct(test_labels),
})

print("\n=== 3) Class Distribution Consistency Across Splits ===")
print(dist_df.to_string())

max_dist_diff = (dist_df.max(axis=1) - dist_df.min(axis=1)).max()
print(f"\nMax class-proportion difference across splits: {max_dist_diff:.2f} percentage points")
print("✅ Stratification looks consistent" if max_dist_diff < 5 else "⚠️ Notable imbalance across splits — check stratify= was used correctly")


# ============================================================
# 4) FINAL SUMMARY
# ============================================================
integrity_summary = {
    "Check": [
        "Train∩Val index overlap", "Train∩Test index overlap", "Val∩Test index overlap",
        "Train-Val near-duplicates", "Train-Test near-duplicates", "Val-Test near-duplicates",
        "Max class-dist difference (pp)",
    ],
    "Result": [
        len(overlap_train_val), len(overlap_train_test), len(overlap_val_test),
        len(dup_train_val), len(dup_train_test), len(dup_val_test),
        round(max_dist_diff, 2),
    ],
    "Status": [
        "✅ Pass" if len(overlap_train_val) == 0 else "⚠️ FAIL",
        "✅ Pass" if len(overlap_train_test) == 0 else "⚠️ FAIL",
        "✅ Pass" if len(overlap_val_test) == 0 else "⚠️ FAIL",
        "✅ Pass" if len(dup_train_val) == 0 else "⚠️ FAIL",
        "✅ Pass" if len(dup_train_test) == 0 else "⚠️ FAIL",
        "✅ Pass" if len(dup_val_test) == 0 else "⚠️ FAIL",
        "✅ Pass" if max_dist_diff < 5 else "⚠️ Review",
    ],
}
integrity_df = pd.DataFrame(integrity_summary)
integrity_df.to_csv("/kaggle/working/split_integrity_check_summary.csv", index=False)
print("\n=== Final Integrity Check Summary ===")
print(integrity_df.to_string(index=False))